# Diagnostic Tests II: Exercises
### Applied Statistical Data Analysis. Prof. Dr. Kristyna Ters | MSc Finance | FHNW

---
> **Instructions:**
> - Work through the exercises in order; each builds on the previous one
> - Fill in your code in the cells marked with `# YOUR CODE HERE`
> - Answer written questions by double-clicking the markdown cell and editing it
> - Run cells with **Shift+Enter**
> - Solutions will be released after the submission deadline

In [ ]:
!pip install yfinance pandas-datareader statsmodels --quiet

import yfinance as yf
import pandas_datareader.data as web
import pandas as pd
import numpy as np
import statsmodels.api as sm
from statsmodels.stats.diagnostic import linear_reset
from statsmodels.stats.stattools import jarque_bera
import matplotlib.pyplot as plt
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor':'white', 'axes.facecolor':'white',
    'axes.spines.top':False, 'axes.spines.right':False,
    'axes.grid':True, 'grid.alpha':0.3, 'font.size':11
})
YELLOW = '#FDE70E'; ORANGE = '#FCB310'; RED = '#C70101'
GREY   = '#4B4B4B'; BLUE = '#0E75FE'; GREEN = '#0B7A3C'
print('✓ Libraries loaded.')

---
# Exercise 1: Which Diagnostic? (Warm-Up, No Code)

For each situation, name the appropriate test or tool (RESET, Jarque-Bera, standardized residuals / event dummies, or one from part I), and state what a rejection would mean.

| # | Situation | Tool | Rejection means |
|---|-----------|------|-----------------|
| a | Residuals form a U-shape against the fitted values | | |
| b | The residual histogram has a tall peak and long tails | | |
| c | Five days dominate the scatter plot visually | | |
| d | Residuals fan out with the fitted value | | |
| e | You fitted a price LEVEL on an index LEVEL and worry about the form | | |
| f | A colleague claims the CAPM beta is driven entirely by March 2020 | | |

**Your answers** (double-click to edit):

| # | Tool | Rejection / finding means |
|---|------|---------------------------|
| a | | |
| b | | |
| c | | |
| d | | |
| e | | |
| f | | |

---
# Exercise 2: The Base Model

Our patient for the whole exercise set: the **Nestlé CAPM** against the SMI.

$$r_{NESN,t} = \beta_0 + \beta_1\, r_{SMI,t} + u_t$$

In [ ]:
STOCK, INDEX = 'NESN.SW', '^SSMI'

# YOUR CODE HERE
# 1. Download STOCK and INDEX 2020-01-01 to 2024-12-31 (auto_adjust=True), compute daily returns
# 2. Fit plain OLS: y = Nestle returns, X = sm.add_constant(SMI returns)
# 3. Print n, beta_hat, R²


---
# Exercise 3: RESET on the Return Regression

Run Ramsey RESET (powers up to 3, F-form) on the Nestlé CAPM.

**Written question:** the test will very likely pass. Why is a *pass* informative here, and what does it say about the CAPM in returns?

In [ ]:
# YOUR CODE HERE
# reset = linear_reset(capm, power=3, use_f=True)
# print F and p, state the decision


---
# Exercise 4: RESET on a Levels Regression

Now regress the Nestlé PRICE level on the SMI INDEX level (both from the same download, before computing returns) and run RESET again.

**Written question:** compare with Exercise 3. What lesson about levels versus returns do the two results teach?

In [ ]:
# YOUR CODE HERE
# 1. lvl = px.dropna(); columns NESN price, SMI level
# 2. m_lvl = OLS(price on constant + index level); print R²
# 3. linear_reset(m_lvl, power=3, use_f=True); print F, p, decision
# 4. Plot residuals vs fitted values: do you see systematic structure?


---
# Exercise 5: Jarque-Bera by Hand

Back to the return CAPM from Exercise 2. Compute skewness S and kurtosis K of the residuals, plug them into

$$JB = n\left[\frac{S^2}{6} + \frac{(K-3)^2}{24}\right],$$

and decide against the χ²(2) critical value 5.99.

**Written question:** which of the two terms dominates, and what feature of return data does that reflect?

In [ ]:
# YOUR CODE HERE
# S = stats.skew(u);  K = stats.kurtosis(u, fisher=False)  → raw kurtosis, normal = 3
# JB = n * (S**2/6 + (K-3)**2/24); compare with 5.99


---
# Exercise 6: Confirm and Visualise

Confirm your hand computation with `jarque_bera`, and produce the two standard pictures: histogram of standardized residuals against the normal density, and a QQ-plot.

In [ ]:
# YOUR CODE HERE
# jb, p, skew, kurt = jarque_bera(capm.resid)
# histogram (density=True) + stats.norm.pdf overlay; stats.probplot for the QQ-plot


---
# Exercise 7: Count and Identify the Outliers

Standardize the residuals, count the days with $|z| > 3$, compare with the count normality predicts, and list the five most extreme days with their dates.

**Written question:** which events do the top dates correspond to? (Think 2020 and, for Swiss stocks, March 2023.)

In [ ]:
# YOUR CODE HERE
# z = capm.resid / capm.resid.std()
# expected = 2 * (1 - stats.norm.cdf(3)) * n
# flagged = z[abs(z) > 3]; list the 5 largest |z| with dates


---
# Exercise 8: The Event-Dummy Robustness Check

Add one 0/1 dummy for each of the three most extreme days and re-estimate with HC1 standard errors. Compare beta, SE and JB with the base model.

**Written question:** state in one sentence what the comparison tells you about the Nestlé beta.

In [ ]:
# YOUR CODE HERE
# events = z.abs().sort_values(ascending=False).head(3).index
# add one column per event: (ret.index == d).astype(float)
# refit with cov_type='HC1'; build a small comparison DataFrame (beta, SE, JB)


---
# Exercise 9: The Honesty Rule, Demonstrated

Now do what one should NOT do: add dummies for EVERY day with $|z| > 2$ and re-estimate. Report how many dummies that takes, the new R², and JB.

**Written question:** the fit improves and JB may finally pass. Why is this bad practice anyway? Give the statistical and the economic argument.

In [ ]:
# YOUR CODE HERE
# days2 = z[abs(z) > 2].index; one dummy per day (this will be MANY columns)
# refit; print number of dummies, R², JB before/after


---
# Exercise 10: The Full Verdict (Written)

Summarise your findings for the Nestlé CAPM in at most six sentences: functional form (Exercises 3 and 4), normality (5 and 6), outliers (7 to 9). For each of the three, state the finding AND the consequence for how you would report results.

---
*Applied Statistical Data Analysis | Prof. Dr. Kristyna Ters | FHNW School of Business | HS 2026*